# ESS Battery Health

# 0. 필수 라이브러리 및 경로 설정
분석에 필요한 필수 라이브러리 로드 및 프로젝트 경로, 배치 파일 경로 설정

In [1]:
import gc
import re
import warnings
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

CURRENT_DIR = Path.cwd().resolve()
ROOT = CURRENT_DIR.parent if CURRENT_DIR.name in ('notebooks', 'test') else CURRENT_DIR

DATA_DIR = ROOT / 'data'
RESULT_DIR = ROOT / 'results'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_FILES = {
    'Batch 1': DATA_DIR / 'batch_1_170512.mat',
    'Batch 2': DATA_DIR / 'batch_2_180220.mat',
    'Batch 3': DATA_DIR / 'batch_3_180412.mat',
}

batch_cell_frames = []

print(f'데이터 경로: {DATA_DIR}')
print(f'결과 저장 경로: {RESULT_DIR}')
for name, path in BATCH_FILES.items():
    print(f'{name}: {path.name} ({path.stat().st_size / 1024**3:.1f} GB)')

데이터 경로: /Users/yujin.je/Workspace/ds-mini/data
결과 저장 경로: /Users/yujin.je/Workspace/ds-mini/results
Batch 1: batch_1_170512.mat (2.8 GB)
Batch 2: batch_2_180220.mat (1.9 GB)
Batch 3: batch_3_180412.mat (3.0 GB)


# 1. 공통 함수 정의
파일 로드, 기초 데이터 확인, 그래프 저장 등 Batch별 반복 작업용 공통 함수 정의 및 재사용

In [2]:
def parse_charging_policy(policy):
    pattern = r'([0-9.]+)C\(([0-9.]+)%\)-([0-9.]+)C'
    match = re.search(pattern, policy)
    if match:
        first_c_rate = float(match.group(1))
        switch_soc = float(match.group(2))
        second_c_rate = float(match.group(3))
    else:
        first_c_rate = np.nan
        switch_soc = np.nan
        second_c_rate = np.nan
    newstructure = int('newstructure' in policy.lower())
    return first_c_rate, switch_soc, second_c_rate, newstructure


def detect_piecewise_knee(one_cell, min_points=30, step=5):
    data = (
        one_cell[['cycle', 'QD']]
        .dropna()
        .sort_values('cycle')
        .copy()
    )
    if len(data) < min_points * 2:
        return {
            'knee_cycle': np.nan,
            'pre_knee_slope': np.nan,
            'post_knee_slope': np.nan,
            'slope_ratio': np.nan,
        }
    data['QD_smooth'] = (
        data['QD']
        .rolling(11, center=True, min_periods=1)
        .median()
    )
    x = data['cycle'].to_numpy()
    y = data['QD_smooth'].to_numpy()
    start = max(min_points, int(len(data) * 0.20))
    end = min(len(data) - min_points, int(len(data) * 0.90))
    best = None
    best_err = np.inf
    for split in range(start, end, step):
        p1 = np.polyfit(x[:split], y[:split], 1)
        p2 = np.polyfit(x[split:], y[split:], 1)
        err = np.sum((y[:split] - np.polyval(p1, x[:split]))**2) + np.sum((y[split:] - np.polyval(p2, x[split:]))**2)
        if err < best_err:
            best_err = err
            best = (split, p1[0], p2[0])
            
    if best is not None:
        split_idx, s1, s2 = best
        knee_c = float(x[split_idx])
        ratio = abs(s2 / s1) if abs(s1) > 1e-9 else np.nan
        return {
            'knee_cycle': knee_c,
            'pre_knee_slope': s1,
            'post_knee_slope': s2,
            'slope_ratio': ratio,
        }
    return {
        'knee_cycle': np.nan,
        'pre_knee_slope': np.nan,
        'post_knee_slope': np.nan,
        'slope_ratio': np.nan,
    }


def load_batch(batch_name, path):
    # MAT 파일 한 개에서 셀 정보, 사이클 요약, ΔQ(V) 곡선을 읽어 반환
    cell_rows = []
    summary_rows = []
    delta_q_curves = {}

    print(f'{batch_name} 로딩 중...')
    with h5py.File(path, 'r') as file:
        batch = file['batch']

        for i in range(batch['cycle_life'].shape[0]):
            cell_id = f'b{batch_name[-1]}c{i}'

            life_values = np.asarray(file[batch['cycle_life'][i, 0]]).reshape(-1)
            cycle_life = float(life_values[0]) if len(life_values) else np.nan

            policy_values = np.asarray(file[batch['policy_readable'][i, 0]]).reshape(-1)
            policy = ''.join(chr(int(value)) for value in policy_values)
            first_c_rate, switch_soc, second_c_rate, newstructure = parse_charging_policy(policy)

            summary = file[batch['summary'][i, 0]]
            cycle = np.asarray(summary['cycle']).squeeze()
            qd = np.asarray(summary['QDischarge']).squeeze()
            qc = np.asarray(summary['QCharge']).squeeze()
            ir = np.asarray(summary['IR']).squeeze()
            tavg = np.asarray(summary['Tavg']).squeeze()
            tmax = np.asarray(summary['Tmax']).squeeze()
            chargetime = np.asarray(summary['chargetime']).squeeze()

            initial_qd = np.nanmedian(qd[(cycle >= 10) & (cycle <= 20)])
            capacity_95_candidates = cycle[(cycle >= 100) & (qd < initial_qd * 0.95)]
            capacity_95_cycle = capacity_95_candidates[0] if len(capacity_95_candidates) else np.nan

            cycles = file[batch['cycles'][i, 0]]
            qd_10 = np.asarray(file[cycles['Qdlin'][9, 0]]).squeeze()
            qd_100 = np.asarray(file[cycles['Qdlin'][99, 0]]).squeeze()
            delta_q = qd_100 - qd_10
            delta_q_curves[cell_id] = delta_q

            # 초기 10~100 사이클 요약 통계 및 early_QD_slope 계산
            early_mask = (cycle >= 10) & (cycle <= 100)
            if early_mask.sum() >= 2:
                early_qd_slope = float(np.polyfit(cycle[early_mask], qd[early_mask], 1)[0])
            else:
                early_qd_slope = np.nan

            cell_rows.append({
                'batch': batch_name,
                'cell_id': cell_id,
                'cycle_life': cycle_life,
                'charging_policy': policy,
                'first_C_rate': first_c_rate,
                'switch_SOC': switch_soc,
                'second_C_rate': second_c_rate,
                'newstructure': newstructure,
                'capacity_95_cycle': capacity_95_cycle,
                'delta_q_min': np.nanmin(delta_q),
                'delta_q_var': np.nanvar(delta_q),
                'early_QD_slope': early_qd_slope,
                'mean_QD': float(np.nanmean(qd[early_mask])) if early_mask.any() else np.nan,
                'std_QD': float(np.nanstd(qd[early_mask])) if early_mask.any() else np.nan,
                'mean_IR': float(np.nanmean(ir[early_mask])) if early_mask.any() else np.nan,
                'mean_Tavg': float(np.nanmean(tavg[early_mask])) if early_mask.any() else np.nan,
                'mean_Tmax': float(np.nanmean(tmax[early_mask])) if early_mask.any() else np.nan,
                'mean_chargetime': float(np.nanmean(chargetime[early_mask])) if early_mask.any() else np.nan,
            })

            for j in range(len(cycle)):
                summary_rows.append({
                    'batch': batch_name,
                    'cell_id': cell_id,
                    'cycle': cycle[j],
                    'cycle_life': cycle_life,
                    'charging_policy': policy,
                    'QD': qd[j],
                    'QC': qc[j],
                    'IR': ir[j],
                    'Tavg': tavg[j],
                    'Tmax': tmax[j],
                    'chargetime': chargetime[j],
                })

    print(f'로드 완료: 셀 {len(cell_rows)}개 / summary {len(summary_rows):,}행')
    return pd.DataFrame(cell_rows), pd.DataFrame(summary_rows), delta_q_curves


In [3]:
def check_batch(batch_name, cell_df, summary_df):
    # 로드한 Batch의 크기, 결측치, 기초 통계, 첫 5행을 출력
    print('=' * 60)
    print(batch_name)
    print('=' * 60)
    print(f'셀 수             : {len(cell_df)}')
    print(f'Cycle Life 결측   : {cell_df.cycle_life.isna().sum()}')
    print(f'충전 정책 수      : {cell_df.charging_policy.nunique()}')
    print(f'Cell DataFrame    : {cell_df.shape}')
    print(f'Summary DataFrame : {summary_df.shape}')
    
    display(cell_df.head())
    display(cell_df[['cycle_life', 'first_C_rate', 'capacity_95_cycle', 'delta_q_min', 'delta_q_var']].describe().round(3))


In [4]:
def save_plot(fig, filename):
    # 그래프를 results 폴더에 저장한 뒤 노트북 화면에도 출력한다.
    save_path = RESULT_DIR / filename
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight')

    plt.close(fig)
    
    print(f'저장 완료: {save_path.relative_to(ROOT)}')


# 2. EDA

Batch 1, 2, 3 순차 로드 및 5대 질문 기반 분석 진행, **관찰 수치 → 해석 → 모델링 시사점** 흐름으로 결과 정리

- Cycle Life 단수명 기준 `<500`, 장수명 기준 `>1,000` 설정 (표본 부족 시 상대적 상·하위 25% 보조 비교)
- 소표본 및 비정규 분포 특성을 고려하여 **Spearman 순위상관계수** 활용
- 충전 정책별 평균과 상관관계는 단순 연관성이며 인과관계가 아니므로, 정책당 표본 수와 `newstructure` 여부 병행 확인
- `capacity_95_cycle`: Cycle 10~20 초기 QD 중앙값 대비 5% 이상 용량 감소 최초 시점으로 정의한 보조 지표
- `knee_cycle`: 전체 QD 열화 곡선을 2구간 piecewise linear regression으로 근사하여 최적 breakpoint 탐지
- 두 지표 모두 전체 수명 관측이 필요한 사후 지표이므로 EDA 및 결과 해석용으로만 한정, 초기 예측 피처에서 제외

### 2-1. Batch 1 EDA

#### 기초 EDA

In [5]:
batch1_cells, batch1_summary, batch1_delta_q = load_batch(
    'Batch 1', BATCH_FILES['Batch 1']
)
check_batch('Batch 1', batch1_cells, batch1_summary)

Batch 1 로딩 중...
로드 완료: 셀 46개 / summary 38,811행
Batch 1
셀 수             : 46
Cycle Life 결측   : 0
충전 정책 수      : 23
Cell DataFrame    : (46, 18)
Summary DataFrame : (38811, 11)


,batch,cell_id,cycle_life,charging_policy,first_C_rate,switch_SOC,second_C_rate,newstructure,capacity_95_cycle,delta_q_min,delta_q_var,early_QD_slope,mean_QD,std_QD,mean_IR,mean_Tavg,mean_Tmax,mean_chargetime
0,Batch 1,b1c0,1190.0,3.6C(80%)-3.6C,3.6,80.0,3.6,0,611.0,-0.008460,0.000010,-0.000314,1.081257,0.048263,0.016643,31.576035,35.353079,13.384324
1,Batch 1,b1c1,1179.0,3.6C(80%)-3.6C,3.6,80.0,3.6,0,NaN,-0.011004,0.000010,-0.000014,1.081534,0.000581,0.016736,31.295828,34.131879,13.372038
2,Batch 1,b1c2,1177.0,3.6C(80%)-3.6C,3.6,80.0,3.6,0,NaN,-0.017216,0.000018,-0.000005,1.085624,0.000458,0.016588,31.442117,34.516218,13.379405
3,Batch 1,b1c3,1226.0,4C(80%)-4C,4.0,80.0,4.0,0,1035.0,-0.018961,0.000036,0.000006,1.085162,0.000751,0.016133,29.908330,30.634385,12.053526
4,Batch 1,b1c4,1227.0,4C(80%)-4C,4.0,80.0,4.0,0,1174.0,-0.013958,0.000023,0.000008,1.083063,0.000766,0.016483,31.428182,34.369298,12.053579


,cycle_life,first_C_rate,capacity_95_cycle,delta_q_min,delta_q_var
count,46.000,46.000,44.000,46.000,46.0
mean,844.717,5.878,632.091,-0.036,0.0
std,184.629,1.203,161.260,0.013,0.0
min,534.000,3.600,255.000,-0.065,0.0
25%,703.250,5.400,535.500,-0.045,0.0
50%,858.500,6.000,637.000,-0.036,0.0
75%,914.250,6.750,722.500,-0.029,0.0
max,1227.000,8.000,1174.000,-0.008,0.0


#### Question 1. Cycle Life 분포는 어떻게 생겼는가?
- 150 ~ 2,300 사이클 Histogram
- 장수명(>1,000)/단수명(<500) 비율 확인
- 이상치 셀 식별 - 왜 유독 짧은가?

In [6]:
valid = batch1_cells[batch1_cells.cycle_life.notna()].copy()
short_count = (valid.cycle_life < 500).sum()
long_count = (valid.cycle_life > 1000).sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(valid.cycle_life, bins=np.arange(150, 2351, 100),
             color='steelblue', edgecolor='white')
axes[0].axvline(500, color='tomato', linestyle='--', label='Short < 500')
axes[0].axvline(1000, color='seagreen', linestyle='--', label='Long > 1,000')
axes[0].set(title='Histogram', xlabel='Cycle Life', ylabel='num of Battery Cell')
axes[0].legend()
axes[1].boxplot(valid.cycle_life, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=.6))
axes[1].set(title='Box Plot', ylabel='Cycle Life', xticks=[])
fig.suptitle('Batch 1 - Cycle Life', fontweight='bold')
save_plot(fig, 'batch_1_q1_cycle_life.png')

print(valid.cycle_life.describe().round(1))
print(f'Short: {short_count}개 ({short_count/len(valid):.1%})')
print(f'Long : {long_count}개 ({long_count/len(valid):.1%})')
display(valid.nsmallest(5, 'cycle_life')[
    ['cell_id', 'cycle_life', 'charging_policy', 'delta_q_var']])

# 최단수명 5개 셀 상세 및 전체 median 비교
features = batch1_cells.copy()
outlier_cols = [
    'cell_id',
    'cycle_life',
    'charging_policy',
    'first_C_rate',
    'switch_SOC',
    'second_C_rate',
    'delta_q_var',
    'delta_q_min',
    'early_QD_slope',
    'mean_IR',
    'mean_Tavg',
    'mean_chargetime',
]
shortest_cells = features.nsmallest(5, 'cycle_life')[outlier_cols]
print('[Batch 1 최단수명 5개 셀]')
display(shortest_cells)

compare_cols = [
    'delta_q_var',
    'delta_q_min',
    'early_QD_slope',
    'mean_IR',
    'mean_Tavg',
    'mean_chargetime',
]
short_ids = shortest_cells['cell_id']
comparison = pd.DataFrame({
    'shortest_5_median': features[features['cell_id'].isin(short_ids)][compare_cols].median(),
    'batch_median': features[compare_cols].median()
})
print('[Batch 1 최단수명 5개 median vs Batch 전체 median]')
display(comparison.round(5))


저장 완료: results/batch_1_q1_cycle_life.png
count      46.0
mean      844.7
std       184.6
min       534.0
25%       703.2
50%       858.5
75%       914.2
max      1227.0
Name: cycle_life, dtype: float64
Short: 0개 (0.0%)
Long : 10개 (21.7%)


,cell_id,cycle_life,charging_policy,delta_q_var
20,b1c20,534.0,5.4C(80%)-5.4C,0.000429
21,b1c21,559.0,5.4C(80%)-5.4C,0.000446
45,b1c45,599.0,8C(35%)-3.6C,0.000376
44,b1c44,616.0,8C(35%)-3.6C,0.000380
38,b1c38,617.0,7C(40%)-3.6C,0.000369


[Batch 1 최단수명 5개 셀]


,cell_id,cycle_life,charging_policy,first_C_rate,switch_SOC,second_C_rate,delta_q_var,delta_q_min,early_QD_slope,mean_IR,mean_Tavg,mean_chargetime
20,b1c20,534.0,5.4C(80%)-5.4C,5.4,80.0,5.4,0.000429,-0.063580,-0.000149,0.015015,32.126770,9.027973
21,b1c21,559.0,5.4C(80%)-5.4C,5.4,80.0,5.4,0.000446,-0.064720,-0.000090,0.014917,33.108984,8.989637
45,b1c45,599.0,8C(35%)-3.6C,8.0,35.0,3.6,0.000376,-0.059404,-0.000023,0.016680,33.019806,10.258618
44,b1c44,616.0,8C(35%)-3.6C,8.0,35.0,3.6,0.000380,-0.058797,-0.000042,0.017418,32.567783,10.257644
38,b1c38,617.0,7C(40%)-3.6C,7.0,40.0,3.6,0.000369,-0.056416,-0.000019,0.016216,33.708385,10.232226


[Batch 1 최단수명 5개 median vs Batch 전체 median]


,shortest_5_median,batch_median
delta_q_var,0.00038,0.00014
delta_q_min,-0.05940,-0.03635
early_QD_slope,-0.00004,-0.00001
mean_IR,0.01622,0.01659
mean_Tavg,33.01981,32.79478
mean_chargetime,10.23223,11.14523


#### Question 2. 열화 곡선 - 방전 용량이 어떻게 감소하는가?
- 사이클 별 Qd 추이 시각화
- 열화 속도가 일정한가, 가속되는가?
- Knee point - 급격한 열화 시작점 탐색

In [7]:
valid_summary = batch1_summary[batch1_summary.cycle_life.notna()].copy()
cell_ids = valid_summary.cell_id.unique()
colors = cm.coolwarm(np.linspace(0, 1, len(cell_ids)))
nominal = valid_summary[valid_summary.cycle.between(10, 20)].groupby('cell_id').QD.median().median()

fig, ax = plt.subplots(figsize=(14, 6))
for color, cell_id in zip(colors, cell_ids):
    one_cell = valid_summary[
        (valid_summary.cell_id == cell_id)
        & valid_summary.QD.between(nominal * .7, nominal * 1.2)
    ]
    ax.plot(one_cell.cycle, one_cell.QD, color=color, linewidth=.8, alpha=.7)
ax.axhline(nominal * .8, color='red', linestyle='--', label='EOL (80%)')
ax.set(title='Batch 1 - QD Curve', xlabel='Cycle', ylabel='QD (Ah)', ylim=(.7, 1.2))
ax.legend()
save_plot(fig, 'batch_1_q2_degradation.png')

print('용량 95% 도달 시점 (capacity_95_cycle) 요약')
display(batch1_cells.loc[batch1_cells.cycle_life.notna(), 'capacity_95_cycle'].describe().round(1))

# 실제 Piecewise Knee 검출 분석
knee_rows = []
for cell_id, one_cell in batch1_summary.groupby('cell_id'):
    result = detect_piecewise_knee(one_cell)
    result['cell_id'] = cell_id
    knee_rows.append(result)
knee_df = pd.DataFrame(knee_rows)
knee_df = knee_df.merge(
    batch1_cells[['cell_id', 'cycle_life']],
    on='cell_id',
    how='left')
valid_knee = knee_df.dropna(
    subset=['knee_cycle', 'cycle_life']).copy()
valid_knee['knee_ratio'] = (
    valid_knee['knee_cycle']
    / valid_knee['cycle_life'])

if len(valid_knee) > 1:
    rho = valid_knee[['knee_cycle', 'cycle_life']].corr(method='spearman').iloc[0, 1]
    print(f'[Batch 1] Knee vs Cycle Life rho: {rho:.3f}')
    print(f'[Batch 1] Median Knee Cycle: {valid_knee.knee_cycle.median():.1f}')
    print(f'[Batch 1] Median Knee Ratio: {valid_knee.knee_ratio.median():.3f}')
    print(f'[Batch 1] Median Pre-Knee Slope: {valid_knee.pre_knee_slope.median():.6f}')
    print(f'[Batch 1] Median Post-Knee Slope: {valid_knee.post_knee_slope.median():.6f}')
    print(f'[Batch 1] Median Slope Ratio: {valid_knee.slope_ratio.median():.2f}x')


저장 완료: results/batch_1_q2_degradation.png
용량 95% 도달 시점 (capacity_95_cycle) 요약


count      44.0
mean      632.1
std       161.3
min       255.0
25%       535.5
50%       637.0
75%       722.5
max      1174.0
Name: capacity_95_cycle, dtype: float64

[Batch 1] Knee vs Cycle Life rho: 0.658
[Batch 1] Median Knee Cycle: 602.0
[Batch 1] Median Knee Ratio: 0.749
[Batch 1] Median Pre-Knee Slope: -0.000069
[Batch 1] Median Post-Knee Slope: -0.000716
[Batch 1] Median Slope Ratio: 9.78x


#### Question 3. ΔQ(V) 곡선 - 초기 사이클에서 차이가 보이는가?
- 사이클 100번 - 사이클 10번의 Q(V) 차이 계산
- 장수명 셀 vs 단수명 셀의 ΔQ 형태 비교
- 이를 구분할 수 있는 통계값으로 피쳐 추출

In [8]:
life_group = batch1_cells[batch1_cells.cycle_life.notna()].copy()
life_group['life_group'] = 'middle'
life_group.loc[life_group.cycle_life < 500, 'life_group'] = 'short'
life_group.loc[life_group.cycle_life > 1000, 'life_group'] = 'long'

if (life_group.life_group == 'short').sum() < 2 or (life_group.life_group == 'long').sum() < 2:
    low, high = life_group.cycle_life.quantile([.25, .75])
    # 기존 absolute label 초기화
    life_group['life_group'] = 'middle'
    life_group.loc[life_group.cycle_life <= low, 'life_group'] = 'short'
    life_group.loc[life_group.cycle_life >= high, 'life_group'] = 'long'
    print(f'절대 기준 표본 부족 → 하위 25%(≤{low:.0f})와 상위 25%(≥{high:.0f}) 비교')

voltage = np.linspace(2, 3.6, 1000)
fig, ax = plt.subplots(figsize=(10, 5))
for group, color in [('short', 'tomato'), ('long', 'seagreen')]:
    ids = life_group.loc[life_group.life_group == group, 'cell_id']
    curves = np.vstack([batch1_delta_q[cell_id] for cell_id in ids])
    ax.plot(voltage, curves.mean(axis=0), color=color, label=f'{group} (n={len(ids)})')
ax.set(title='Batch 1 - Delta Q(V)', xlabel='Voltage (V)',
       ylabel='Qd(100) - Qd(10) (Ah)')
ax.legend()
save_plot(fig, 'batch_1_q3_delta_q.png')
display(life_group.groupby('life_group')[['delta_q_min', 'delta_q_var']].median().round(6))


절대 기준 표본 부족 → 하위 25%(≤703)와 상위 25%(≥914) 비교
저장 완료: results/batch_1_q3_delta_q.png


,delta_q_min,delta_q_var
life_group,,
long,-0.026843,0.000076
middle,-0.033990,0.000126
short,-0.048927,0.000268


#### Question 4. 충전 조건 (C-rate)과 수명의 관계는?
- 충전 프로토콜별 평균 수명 비교
- 고속 충전 셀이 정말 수명이 짧은가?
- 충전 전류 패턴과 열화 속도 상관 분석

In [9]:
valid = batch1_cells[batch1_cells.cycle_life.notna()].copy()
features = batch1_cells.copy()

policy_summary = (
    features
    .groupby('charging_policy')['cycle_life']
    .agg([
        'count',
        'mean',
        'median',
        'std',
        'min',
        'max'
    ])
    .sort_values('median', ascending=False)
)
policy_summary.to_csv(RESULT_DIR / 'batch_1_policy_summary.csv')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar(range(len(policy_summary)), policy_summary['mean'], color='steelblue', alpha=.75)
axes[0].set_xticks(range(len(policy_summary)))
axes[0].set_xticklabels(policy_summary.index, rotation=45, ha='right', fontsize=8)
axes[0].set(title='Average Cycle Life by Policy', ylabel='mean(Cycle Life)')
axes[1].scatter(valid.first_C_rate, valid.cycle_life,
                color='steelblue', alpha=.7, edgecolors='white')
corr = valid.first_C_rate.corr(valid.cycle_life, method='spearman')
axes[1].set(title=f'First C-rate vs Cycle Life (rho={corr:.2f})',
            xlabel='First C-rate', ylabel='Cycle Life')
fig.suptitle('Batch 1 - Charging Policy', fontweight='bold')
save_plot(fig, 'batch_1_q4_charge_policy.png')

print('[Batch 1 Charging Policy Summary]')
display(policy_summary.round(1))

# C-rate / SOC 분석
charge_features = [
    'first_C_rate',
    'switch_SOC',
    'second_C_rate',
]
target_features = [
    'cycle_life',
    'delta_q_var',
    'early_QD_slope',
]
charge_corr = (
    features[
        charge_features + target_features
    ]
    .corr(method='spearman')
    .loc[
        charge_features,
        target_features
    ]
)
print('[Batch 1 Charging Condition Correlation]')
display(charge_corr.round(3))

# newstructure 분석
structure_summary = (
    features
    .groupby('newstructure')
    .agg(
        n=('cell_id', 'count'),
        median_cycle_life=('cycle_life', 'median'),
        mean_cycle_life=('cycle_life', 'mean'),
        median_delta_q_var=('delta_q_var', 'median'),
        median_early_qd_slope=('early_QD_slope', 'median'),
    )
)
print('[Batch 1 newstructure Summary]')
display(structure_summary.round(4))


저장 완료: results/batch_1_q4_charge_policy.png
[Batch 1 Charging Policy Summary]


,count,mean,median,std,min,max
charging_policy,,,,,,
4C(80%)-4C,2,1226.5,1226.5,0.7,1226.0,1227.0
3.6C(80%)-3.6C,3,1182.0,1179.0,7.0,1177.0,1190.0
4.4C(80%)-4.4C,1,1074.0,1074.0,NaN,1074.0,1074.0
8C(15%)-3.6C,2,1008.5,1008.5,60.1,966.0,1051.0
5.4C(40%)-3.6C,2,966.5,966.5,123.7,879.0,1054.0
6C(30%)-3.6C,2,952.5,952.5,87.0,891.0,1014.0
6C(40%)-3C,2,935.5,935.5,115.3,854.0,1017.0
5.4C(50%)-3.6C,2,899.5,899.5,3.5,897.0,902.0
6C(50%)-3C,2,888.5,888.5,40.3,860.0,917.0


[Batch 1 Charging Condition Correlation]


,cycle_life,delta_q_var,early_QD_slope
first_C_rate,-0.483,0.540,-0.328
switch_SOC,0.185,-0.198,0.072
second_C_rate,0.047,-0.066,-0.050


[Batch 1 newstructure Summary]


,n,median_cycle_life,mean_cycle_life,median_delta_q_var,median_early_qd_slope
newstructure,,,,,
0,46,858.5,844.7174,0.0001,-0.0


#### Question 5. 상관관계 - 어떤 신호가 수명과 연관되어 있는가?
- 초기 사이클 피켜들과 Cycle Life 상관 계수 확인
- 가장 강한 관계 식별
- 멀티클리니어리티 문제 확인

In [10]:
valid_summary = batch1_summary[batch1_summary.cycle_life.notna()].copy()
early_features = valid_summary[valid_summary.cycle.between(10, 100)].groupby('cell_id').agg(
    cycle_life=('cycle_life', 'first'),
    mean_QD=('QD', 'mean'),
    std_QD=('QD', 'std'),
    mean_IR=('IR', 'mean'),
    mean_Tavg=('Tavg', 'mean'),
    mean_Tmax=('Tmax', 'mean'),
    mean_chargetime=('chargetime', 'mean'),
)
extra_features = batch1_cells.set_index('cell_id')[['delta_q_min', 'delta_q_var', 'first_C_rate']]
early_features = early_features.join(extra_features)
corr_matrix = early_features.corr(method='spearman')
early_features.to_csv(RESULT_DIR / 'batch_1_early_features.csv')
corr_matrix.to_csv(RESULT_DIR / 'batch_1_correlation.csv')

fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(image, ax=ax)
labels = corr_matrix.columns.tolist()
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticklabels(labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        value = corr_matrix.iloc[i, j]
        ax.text(j, i, f'{value:.2f}', ha='center', va='center', fontsize=8,
                color='white' if abs(value) > .5 else 'black')
ax.set_title('Batch 1 - Early 100 Cycle Correlation', fontweight='bold')
save_plot(fig, 'batch_1_q5_correlation.png')

print('[Cycle Life 상관 순위]')
display(corr_matrix.cycle_life.drop('cycle_life').sort_values(key=abs, ascending=False).round(3))

print('[피처끼리 |rho| > 0.9인 조합: 다중공선성 후보]')
pairs = []
feature_corr = corr_matrix.drop(index='cycle_life', columns='cycle_life')
for i in range(len(feature_corr.columns)):
    for j in range(i + 1, len(feature_corr.columns)):
        value = feature_corr.iloc[i, j]
        if abs(value) > .9:
            pairs.append((feature_corr.columns[i], feature_corr.columns[j], round(value, 3)))
display(pd.DataFrame(pairs, columns=['feature_1', 'feature_2', 'rho']))


저장 완료: results/batch_1_q5_correlation.png
[Cycle Life 상관 순위]


delta_q_var       -0.871
delta_q_min        0.850
mean_chargetime    0.617
first_C_rate      -0.483
mean_Tavg         -0.371
mean_Tmax         -0.322
std_QD            -0.259
mean_QD            0.203
mean_IR            0.136
Name: cycle_life, dtype: float64

[피처끼리 |rho| > 0.9인 조합: 다중공선성 후보]


,feature_1,feature_2,rho
0,mean_Tavg,mean_Tmax,0.953
1,delta_q_min,delta_q_var,-0.987


#### Question별 핵심 해석 및 시사점

| Question | 확인한 핵심 내용 | 해석 | 모델링 시사점 |
|---|---|---|---|
| **Q1. Cycle Life 분포** | • 46개 셀 전체 유효 (결측 없음)<br>• 중앙값 **858.5 cycle**, 범위 **534~1,227** (단수명 `<500` 0%, 장수명 `>1,000` 21.7%)<br>• 최단수명 5개 셀(`b1c20`, `b1c21` 등)은 `delta_q_var`가 중앙값 대비 약 **2.7배** 높음 | • 단수명군 부재로 절대 기준 이진분류 적용 불가<br>• 최단수명 셀 공통 정책은 표본 2개로 원인 단정 불가 | • 연속 수명 보존을 위한 **회귀(Regression)** 모델링 채택<br>• 극단 셀 제거 없이 오차 분석 대상 유지, ΔQ와 충전 조건 결합 검토 |
| **Q2. 열화 곡선과 Knee** | • 초기 완만 구간 경과 후 급격한 용량 감소 전환<br>• 용량 95% 도달 중앙값 **637 cycle**, Piecewise Knee 중앙값 **602 cycle** (수명 상관 0.658)<br>• Knee 발생 시점은 전체 수명의 약 **74.9%** 지점, Knee 이후 열화 기울기 절댓값 약 **9.78배** 가속 | • 장수명 셀일수록 급격한 용량 저하 시점 지연, 수명 후반부 가속 열화 패턴 | • Knee는 사후 메커니즘 설명용으로 한정, 초기 예측 피처 제외<br>• 초기 100 사이클 내 미세 열화 신호 포착 필요 |
| **Q3. ΔQ(V)** | • 하위 25% vs 상위 25% 비교<br>• 상대 단수명군 `delta_q_var` 중앙값은 장수명군의 약 **3.5배** (0.000268 vs 0.000076)<br>• 2.4~3.0 V (특히 2.6 V 부근)에서 곡선 격차 최대 | • Cycle 100 시점에 이미 단수명 셀의 방전 곡선 변화 폭 현저히 큼<br>• 1,000개 전압 포인트 대신 곡선의 깊이·변동성 통계량이 수명 차이 대변 | • `log10(delta_q_var)`를 1순위 핵심 피처로 선정<br>• `delta_q_min`은 대체 후보로 유지하되 동시 투입 지양 (다중공선성 방지) |
| **Q4. 충전 조건** | • 정책별 평균 수명 **546.5~1,226.5 cycle**<br>• 수명 상관: 첫 C-rate **-0.483**, 전환 SOC **+0.185**, 두 번째 C-rate **+0.047**<br>• 정책 대부분 표본 1~3개로 소표본, 비단조 관계 | • 첫 C-rate 단독으로는 정책 효과 설명 불가<br>• 전환 SOC, 두 번째 C-rate 복합 작용 및 소표본 편향 혼재 | • 정책명 one-hot 대신 세부 수치(첫 C-rate, 전환 SOC, 두 번째 C-rate)로 분해<br>• 정책별 그룹 분할(GroupKFold)로 데이터 누수 방지 |
| **Q5. 상관관계** | • 수명 상관: `delta_q_var` **-0.871**, `delta_q_min` **+0.850**, `mean_chargetime` **+0.617** 순<br>• 피처 간 강한 다중공선성 (`delta_q_min`–`delta_q_var` -0.987, `mean_Tavg`–`mean_Tmax` 0.953) | • ΔQ 통계량의 설명력 최우수, 동일 정보 중복 피처군 존재 확인 | • 대표 피처 선별 및 Ridge/Elastic Net 규제 적용<br>• 충전시간은 타 배치 방향성 확인 후 보조 피처 활용 |

**Batch 1 결론:**  
초기 ΔQ 변화가 가장 지배적인 수명 신호, 단일 정책/단일 C-rate의 직접 인과 해석 지양 및 복합 피처 분해 필요

In [11]:
batch_cell_frames.append(batch1_cells.copy())
del batch1_cells, batch1_summary, batch1_delta_q
gc.collect()
print('큰 시계열 데이터 메모리 해제 완료')

큰 시계열 데이터 메모리 해제 완료


### 2-2. Batch 2 EDA

#### 기초 EDA

In [12]:
batch2_cells, batch2_summary, batch2_delta_q = load_batch(
    'Batch 2', BATCH_FILES['Batch 2']
)
check_batch('Batch 2', batch2_cells, batch2_summary)

Batch 2 로딩 중...
로드 완료: 셀 47개 / summary 26,904행
Batch 2
셀 수             : 47
Cycle Life 결측   : 8
충전 정책 수      : 20
Cell DataFrame    : (47, 18)
Summary DataFrame : (26904, 11)


,batch,cell_id,cycle_life,charging_policy,first_C_rate,switch_SOC,second_C_rate,newstructure,capacity_95_cycle,delta_q_min,delta_q_var,early_QD_slope,mean_QD,std_QD,mean_IR,mean_Tavg,mean_Tmax,mean_chargetime
0,Batch 2,b2c0,477.0,5.2C(58%)-4C,5.20,58.0,4.00,0,378.0,-0.055061,0.000326,0.000010,1.105192,0.003194,0.017336,31.819666,35.404462,10.173231
1,Batch 2,b2c1,491.0,5.6C(26%)-4.5C,5.60,26.0,4.50,0,329.0,-0.066115,0.000503,-0.000032,1.104775,0.003515,0.017515,33.614773,36.498762,10.196932
2,Batch 2,b2c2,424.0,5.2C(50%)-4.25C,5.20,50.0,4.25,0,345.0,-0.052718,0.000347,-0.000006,1.074653,0.011718,0.016445,33.171563,37.454011,10.175011
3,Batch 2,b2c3,499.0,4.65C(44%)-5C,4.65,44.0,5.00,0,366.0,-0.055868,0.000319,0.000075,1.101804,0.002936,0.015913,33.938658,38.788881,10.173155
4,Batch 2,b2c4,444.0,4.65C(69%)-6C,4.65,69.0,6.00,0,303.0,-0.070297,0.000585,-0.000066,1.085634,0.003180,0.016138,33.038902,37.443013,10.232925


,cycle_life,first_C_rate,capacity_95_cycle,delta_q_min,delta_q_var
count,39.000,43.000,44.000,47.000,47.000
mean,565.744,4.963,422.136,-0.052,0.000
std,222.202,0.660,193.581,0.024,0.000
min,392.000,3.600,100.000,-0.136,0.000
25%,439.500,4.800,309.500,-0.065,0.000
50%,472.000,5.200,354.000,-0.053,0.000
75%,508.500,5.400,419.000,-0.032,0.000
max,1186.000,6.000,888.000,-0.007,0.002


#### Question 1. Cycle Life 분포는 어떻게 생겼는가?
- 150 ~ 2,300 사이클 Histogram
- 장수명(>1,000)/단수명(<500) 비율 확인
- 이상치 셀 식별 - 왜 유독 짧은가?

In [13]:
valid = batch2_cells[batch2_cells.cycle_life.notna()].copy()
short_count = (valid.cycle_life < 500).sum()
long_count = (valid.cycle_life > 1000).sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(valid.cycle_life, bins=np.arange(150, 2351, 100),
             color='steelblue', edgecolor='white')
axes[0].axvline(500, color='tomato', linestyle='--', label='Short < 500')
axes[0].axvline(1000, color='seagreen', linestyle='--', label='Long > 1,000')
axes[0].set(title='Histogram', xlabel='Cycle Life', ylabel='num of Battery Cell')
axes[0].legend()
axes[1].boxplot(valid.cycle_life, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=.6))
axes[1].set(title='Box Plot', ylabel='Cycle Life', xticks=[])
fig.suptitle('Batch 2 - Cycle Life', fontweight='bold')
save_plot(fig, 'batch_2_q1_cycle_life.png')

print(valid.cycle_life.describe().round(1))
print(f'Short: {short_count}개 ({short_count/len(valid):.1%})')
print(f'Long : {long_count}개 ({long_count/len(valid):.1%})')
display(valid.nsmallest(5, 'cycle_life')[
    ['cell_id', 'cycle_life', 'charging_policy', 'delta_q_var']])

# 최단수명 5개 셀 상세 및 전체 median 비교
features = valid.copy()
outlier_cols = [
    'cell_id',
    'cycle_life',
    'charging_policy',
    'first_C_rate',
    'switch_SOC',
    'second_C_rate',
    'delta_q_var',
    'delta_q_min',
    'early_QD_slope',
    'mean_IR',
    'mean_Tavg',
    'mean_chargetime',
]
shortest_cells = features.nsmallest(5, 'cycle_life')[outlier_cols]
print('[Batch 2 최단수명 5개 셀]')
display(shortest_cells)

compare_cols = [
    'delta_q_var',
    'delta_q_min',
    'early_QD_slope',
    'mean_IR',
    'mean_Tavg',
    'mean_chargetime',
]
short_ids = shortest_cells['cell_id']
comparison = pd.DataFrame({
    'shortest_5_median': features[features['cell_id'].isin(short_ids)][compare_cols].median(),
    'batch_median': features[compare_cols].median()
})
print('[Batch 2 최단수명 5개 median vs Batch 전체 median]')
display(comparison.round(5))


저장 완료: results/batch_2_q1_cycle_life.png
count      39.0
mean      565.7
std       222.2
min       392.0
25%       439.5
50%       472.0
75%       508.5
max      1186.0
Name: cycle_life, dtype: float64
Short: 28개 (71.8%)
Long : 3개 (7.7%)


,cell_id,cycle_life,charging_policy,delta_q_var
19,b2c19,392.0,6C(60%)-3C,0.000511
6,b2c6,393.0,3.6C(9%)-5C,0.000295
15,b2c15,396.0,3.6C(9%)-5C,0.000305
21,b2c21,408.0,6C(60%)-3C,0.000471
30,b2c30,412.0,5.6C(26%)-4.5C,0.000653


[Batch 2 최단수명 5개 셀]


,cell_id,cycle_life,charging_policy,first_C_rate,switch_SOC,second_C_rate,delta_q_var,delta_q_min,early_QD_slope,mean_IR,mean_Tavg,mean_chargetime
19,b2c19,392.0,6C(60%)-3C,6.0,60.0,3.0,0.000511,-0.065652,0.000012,0.016723,32.772474,10.173628
6,b2c6,393.0,3.6C(9%)-5C,3.6,9.0,5.0,0.000295,-0.049872,0.000058,0.016015,33.817061,10.179308
15,b2c15,396.0,3.6C(9%)-5C,3.6,9.0,5.0,0.000305,-0.050935,0.000089,0.016252,32.858335,10.172959
21,b2c21,408.0,6C(60%)-3C,6.0,60.0,3.0,0.000471,-0.064468,0.000039,0.017203,33.620469,10.174718
30,b2c30,412.0,5.6C(26%)-4.5C,5.6,26.0,4.5,0.000653,-0.075523,-0.000089,0.016826,33.819051,53.311883


[Batch 2 최단수명 5개 median vs Batch 전체 median]


,shortest_5_median,batch_median
delta_q_var,0.00047,0.00033
delta_q_min,-0.06447,-0.05417
early_QD_slope,0.00004,0.00002
mean_IR,0.01672,0.01629
mean_Tavg,33.62047,32.99916
mean_chargetime,10.17472,10.17488


#### Question 2. 열화 곡선 - 방전 용량이 어떻게 감소하는가?
- 사이클 별 Qd 추이 시각화
- 열화 속도가 일정한가, 가속되는가?
- Knee point - 급격한 열화 시작점 탐색

In [14]:
valid_summary = batch2_summary[batch2_summary.cycle_life.notna()].copy()
cell_ids = valid_summary.cell_id.unique()
colors = cm.coolwarm(np.linspace(0, 1, len(cell_ids)))
nominal = valid_summary[valid_summary.cycle.between(10, 20)].groupby('cell_id').QD.median().median()

fig, ax = plt.subplots(figsize=(14, 6))
for color, cell_id in zip(colors, cell_ids):
    one_cell = valid_summary[
        (valid_summary.cell_id == cell_id)
        & valid_summary.QD.between(nominal * .7, nominal * 1.2)
    ]
    ax.plot(one_cell.cycle, one_cell.QD, color=color, linewidth=.8, alpha=.7)
ax.axhline(nominal * .8, color='red', linestyle='--', label='EOL (80%)')
ax.set(title='Batch 2 - QD Curve', xlabel='Cycle', ylabel='QD (Ah)', ylim=(.7, 1.2))
ax.legend()
save_plot(fig, 'batch_2_q2_degradation.png')

print('용량 95% 도달 시점 (capacity_95_cycle) 요약')
display(batch2_cells.loc[batch2_cells.cycle_life.notna(), 'capacity_95_cycle'].describe().round(1))

# 실제 Piecewise Knee 검출 분석
knee_rows = []
for cell_id, one_cell in batch2_summary.groupby('cell_id'):
    result = detect_piecewise_knee(one_cell)
    result['cell_id'] = cell_id
    knee_rows.append(result)
knee_df = pd.DataFrame(knee_rows)
knee_df = knee_df.merge(
    batch2_cells[['cell_id', 'cycle_life']],
    on='cell_id',
    how='left')
valid_knee = knee_df.dropna(
    subset=['knee_cycle', 'cycle_life']).copy()
valid_knee['knee_ratio'] = (
    valid_knee['knee_cycle']
    / valid_knee['cycle_life'])

if len(valid_knee) > 1:
    rho = valid_knee[['knee_cycle', 'cycle_life']].corr(method='spearman').iloc[0, 1]
    print(f'[Batch 2] Knee vs Cycle Life rho: {rho:.3f}')
    print(f'[Batch 2] Median Knee Cycle: {valid_knee.knee_cycle.median():.1f}')
    print(f'[Batch 2] Median Knee Ratio: {valid_knee.knee_ratio.median():.3f}')
    print(f'[Batch 2] Median Pre-Knee Slope: {valid_knee.pre_knee_slope.median():.6f}')
    print(f'[Batch 2] Median Post-Knee Slope: {valid_knee.post_knee_slope.median():.6f}')
    print(f'[Batch 2] Median Slope Ratio: {valid_knee.slope_ratio.median():.2f}x')


저장 완료: results/batch_2_q2_degradation.png
용량 95% 도달 시점 (capacity_95_cycle) 요약


count     39.0
mean     431.9
std      181.9
min      234.0
25%      314.0
50%      366.0
75%      419.0
max      888.0
Name: capacity_95_cycle, dtype: float64

[Batch 2] Knee vs Cycle Life rho: 0.956
[Batch 2] Median Knee Cycle: 360.0
[Batch 2] Median Knee Ratio: 0.780
[Batch 2] Median Pre-Knee Slope: -0.000128
[Batch 2] Median Post-Knee Slope: -0.001633
[Batch 2] Median Slope Ratio: 11.54x


#### Question 3. ΔQ(V) 곡선 - 초기 사이클에서 차이가 보이는가?
- 사이클 100번 - 사이클 10번의 Q(V) 차이 계산
- 장수명 셀 vs 단수명 셀의 ΔQ 형태 비교
- 이를 구분할 수 있는 통계값으로 피쳐 추출

In [15]:
life_group = batch2_cells[batch2_cells.cycle_life.notna()].copy()
life_group['life_group'] = 'middle'
life_group.loc[life_group.cycle_life < 500, 'life_group'] = 'short'
life_group.loc[life_group.cycle_life > 1000, 'life_group'] = 'long'

if (life_group.life_group == 'short').sum() < 2 or (life_group.life_group == 'long').sum() < 2:
    low, high = life_group.cycle_life.quantile([.25, .75])
    # 기존 absolute label 초기화
    life_group['life_group'] = 'middle'
    life_group.loc[life_group.cycle_life <= low, 'life_group'] = 'short'
    life_group.loc[life_group.cycle_life >= high, 'life_group'] = 'long'
    print(f'절대 기준 표본 부족 → 하위 25%(≤{low:.0f})와 상위 25%(≥{high:.0f}) 비교')

voltage = np.linspace(2, 3.6, 1000)
fig, ax = plt.subplots(figsize=(10, 5))
for group, color in [('short', 'tomato'), ('long', 'seagreen')]:
    ids = life_group.loc[life_group.life_group == group, 'cell_id']
    curves = np.vstack([batch2_delta_q[cell_id] for cell_id in ids])
    ax.plot(voltage, curves.mean(axis=0), color=color, label=f'{group} (n={len(ids)})')
ax.set(title='Batch 2 - Delta Q(V)', xlabel='Voltage (V)',
       ylabel='Qd(100) - Qd(10) (Ah)')
ax.legend()
save_plot(fig, 'batch_2_q3_delta_q.png')
display(life_group.groupby('life_group')[['delta_q_min', 'delta_q_var']].median().round(6))


저장 완료: results/batch_2_q3_delta_q.png


,delta_q_min,delta_q_var
life_group,,
long,-0.019165,0.000053
middle,-0.027066,0.000070
short,-0.056180,0.000358


#### Question 4. 충전 조건 (C-rate)과 수명의 관계는?
- 충전 프로토콜별 평균 수명 비교
- 고속 충전 셀이 정말 수명이 짧은가?
- 충전 전류 패턴과 열화 속도 상관 분석

In [16]:
valid = batch2_cells[batch2_cells.cycle_life.notna()].copy()
features = valid.copy()

policy_summary = (
    features
    .groupby('charging_policy')['cycle_life']
    .agg([
        'count',
        'mean',
        'median',
        'std',
        'min',
        'max'
    ])
    .sort_values('median', ascending=False)
)
policy_summary.to_csv(RESULT_DIR / 'batch_2_policy_summary.csv')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar(range(len(policy_summary)), policy_summary['mean'], color='steelblue', alpha=.75)
axes[0].set_xticks(range(len(policy_summary)))
axes[0].set_xticklabels(policy_summary.index, rotation=45, ha='right', fontsize=8)
axes[0].set(title='Average Cycle Life by Policy', ylabel='mean(Cycle Life)')
axes[1].scatter(valid.first_C_rate, valid.cycle_life,
                color='steelblue', alpha=.7, edgecolors='white')
corr = valid.first_C_rate.corr(valid.cycle_life, method='spearman')
axes[1].set(title=f'First C-rate vs Cycle Life (rho={corr:.2f})',
            xlabel='First C-rate', ylabel='Cycle Life')
fig.suptitle('Batch 2 - Charging Policy', fontweight='bold')
save_plot(fig, 'batch_2_q4_charge_policy.png')

print('[Batch 2 Charging Policy Summary]')
display(policy_summary.round(1))

# C-rate / SOC 분석
charge_features = [
    'first_C_rate',
    'switch_SOC',
    'second_C_rate',
]
target_features = [
    'cycle_life',
    'delta_q_var',
    'early_QD_slope',
]
charge_corr = (
    features[
        charge_features + target_features
    ]
    .corr(method='spearman')
    .loc[
        charge_features,
        target_features
    ]
)
print('[Batch 2 Charging Condition Correlation]')
display(charge_corr.round(3))

# newstructure 분석
structure_summary = (
    features
    .groupby('newstructure')
    .agg(
        n=('cell_id', 'count'),
        median_cycle_life=('cycle_life', 'median'),
        mean_cycle_life=('cycle_life', 'mean'),
        median_delta_q_var=('delta_q_var', 'median'),
        median_early_qd_slope=('early_QD_slope', 'median'),
    )
)
print('[Batch 2 newstructure Summary]')
display(structure_summary.round(4))


저장 완료: results/batch_2_q4_charge_policy.png
[Batch 2 Charging Policy Summary]


,count,mean,median,std,min,max
charging_policy,,,,,,
5.6C(26%)-4.5C-newstructure,3,991.3,997.0,197.6,791.0,1186.0
5.2C(58%)-4C-newstructure,3,961.7,904.0,157.6,841.0,1140.0
4.8C(80%)-4.8C-newstructure,3,871.7,809.0,137.2,777.0,1029.0
4.8C(80%)-4.8C,5,483.6,492.0,30.3,437.0,514.0
5.2C(58%)-4C,4,477.8,483.0,18.5,452.0,493.0
4.65C(44%)-5C,2,482.5,482.5,23.3,466.0,499.0
4.65C(69%)-6C,2,452.0,452.0,11.3,444.0,460.0
5.2C(50%)-4.25C,5,454.0,449.0,20.9,424.0,474.0
5.6C(26%)-4.5C,6,448.5,446.0,29.1,412.0,491.0


[Batch 2 Charging Condition Correlation]


,cycle_life,delta_q_var,early_QD_slope
first_C_rate,0.055,-0.028,-0.064
switch_SOC,0.288,-0.157,0.014
second_C_rate,-0.119,0.202,-0.127


[Batch 2 newstructure Summary]


,n,median_cycle_life,mean_cycle_life,median_delta_q_var,median_early_qd_slope
newstructure,,,,,
0,30,451.0,453.0000,0.0004,0.0000
1,9,904.0,941.5556,0.0001,-0.0001


#### Question 5. 상관관계 - 어떤 신호가 수명과 연관되어 있는가?
- 초기 사이클 피켜들과 Cycle Life 상관 계수 확인
- 가장 강한 관계 식별
- 멀티클리니어리티 문제 확인

In [17]:
valid_summary = batch2_summary[batch2_summary.cycle_life.notna()].copy()
early_features = valid_summary[valid_summary.cycle.between(10, 100)].groupby('cell_id').agg(
    cycle_life=('cycle_life', 'first'),
    mean_QD=('QD', 'mean'),
    std_QD=('QD', 'std'),
    mean_IR=('IR', 'mean'),
    mean_Tavg=('Tavg', 'mean'),
    mean_Tmax=('Tmax', 'mean'),
    mean_chargetime=('chargetime', 'mean'),
)
extra_features = batch2_cells.set_index('cell_id')[['delta_q_min', 'delta_q_var', 'first_C_rate']]
early_features = early_features.join(extra_features)
corr_matrix = early_features.corr(method='spearman')
early_features.to_csv(RESULT_DIR / 'batch_2_early_features.csv')
corr_matrix.to_csv(RESULT_DIR / 'batch_2_correlation.csv')

fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(image, ax=ax)
labels = corr_matrix.columns.tolist()
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticklabels(labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        value = corr_matrix.iloc[i, j]
        ax.text(j, i, f'{value:.2f}', ha='center', va='center', fontsize=8,
                color='white' if abs(value) > .5 else 'black')
ax.set_title('Batch 2 - Early 100 Cycle Correlation', fontweight='bold')
save_plot(fig, 'batch_2_q5_correlation.png')

print('[Cycle Life 상관 순위]')
display(corr_matrix.cycle_life.drop('cycle_life').sort_values(key=abs, ascending=False).round(3))

print('[피처끼리 |rho| > 0.9인 조합: 다중공선성 후보]')
pairs = []
feature_corr = corr_matrix.drop(index='cycle_life', columns='cycle_life')
for i in range(len(feature_corr.columns)):
    for j in range(i + 1, len(feature_corr.columns)):
        value = feature_corr.iloc[i, j]
        if abs(value) > .9:
            pairs.append((feature_corr.columns[i], feature_corr.columns[j], round(value, 3)))
display(pd.DataFrame(pairs, columns=['feature_1', 'feature_2', 'rho']))


저장 완료: results/batch_2_q5_correlation.png
[Cycle Life 상관 순위]


delta_q_var       -0.709
delta_q_min        0.661
mean_chargetime   -0.391
mean_IR           -0.343
mean_Tavg          0.172
mean_Tmax         -0.126
mean_QD            0.095
std_QD            -0.058
first_C_rate       0.055
Name: cycle_life, dtype: float64

[피처끼리 |rho| > 0.9인 조합: 다중공선성 후보]


,feature_1,feature_2,rho
0,delta_q_min,delta_q_var,-0.985


#### Question별 핵심 해석 및 시사점

| Question | 확인한 핵심 내용 | 해석 | 모델링 시사점 |
|---|---|---|---|
| **Q1. Cycle Life 분포** | • 39개 셀 유효 (8개 결측 제외)<br>• 중앙값 **472.0 cycle**, 범위 **392~1,186** (단수명 `<500` 71.8%, 장수명 `>1,000` 7.7%)<br>• 최단수명 5개 셀은 `delta_q_var` 및 평균 온도가 배치 중앙값 대비 높음 | • 단수명 중심의 극단적 Target shift 발생 (Batch 1 대비 중앙값 45% 감소)<br>• 무작위 분할(Random split) 시 일반화 성능 과대평가 위험 | • 결측 셀 지도학습 제외<br>• Batch 1 학습 모델의 분포 이동 대응력 평가를 위한 **외부 테스트셋**으로 활용 |
| **Q2. 열화 곡선과 Knee** | • 다수 셀이 400~600 cycle에서 조기 급격 열화<br>• 용량 95% 도달 중앙값 **366 cycle**, Piecewise Knee 중앙값 **360 cycle** (수명 상관 0.956)<br>• Knee 발생 시점은 전체 수명의 약 **78.0%** 지점, Knee 이후 열화 기울기 절댓값 약 **11.54배** 가속 | • 조기 열화 배치에서도 수명의 약 78% 시점 Knee 발생 패턴 일관성 유지 | • 열화 시점의 극단적 차이 확인<br>• Knee 피처는 누수로 제외, 초기 ΔQ 신호가 조기 열화 전조를 사전 감지하는지 검증 |
| **Q3. ΔQ(V)** | • 단수명 28개 vs 장수명 3개 비교<br>• 단수명군 `delta_q_var` 중앙값은 장수명군의 약 **6.8배** (0.000358 vs 0.000053)<br>• 2.6 V 부근에서 곡선 간격 최대 | • 불균형 표본 속에서도 Batch 1과 동일한 "ΔQ 변화 클수록 단수명" 패턴 재현 | • Batch 무관 핵심 피처로 `delta_q_var` 채택<br>• 소수 장수명 셀 오차 별도 점검 |
| **Q4. 충전 조건** | • 수명 상관: 첫 C-rate **+0.055**, 전환 SOC **+0.288**, 두 번째 C-rate **-0.119**<br>• `newstructure` 9개 셀의 중앙값은 904.0 cycle로 기존 셀(451.0 cycle) 대비 수명 약 2배 | • Batch 1의 음의 상관 미재현 ("고속 충전=단수명" 가설 기각)<br>• 셀 구조(`newstructure`), 전환 SOC, 2단계 C-rate의 복합 교란 효과 | • `first_C_rate` 단독 사용 지양, 구조 여부·전환 SOC·2차 C-rate 분해<br>• 구조 효과는 인과가 아닌 보정 변수로 취급 |
| **Q5. 상관관계** | • 수명 상관: `delta_q_var` **-0.709**, `delta_q_min` **+0.661**, `mean_chargetime` **-0.391**, `mean_IR` **-0.343** 순<br>• `delta_q_min`–`delta_q_var` 상관 -0.985 | • ΔQ는 Batch 1과 동일 방향 유지, 충전시간은 반대 방향 역전<br>• 충전시간 및 IR은 배치 고유 환경 영향에 민감한 보조 신호 | • 배치 간 일관성이 검증된 `delta_q_var` 최우선 적용<br>• 기타 피처는 ablation 검증 및 규제 선형 모델로 제어 |

**Batch 2 결론:**  
수명 분포와 충전 조건 관계는 Batch 1과 상이하나 초기 ΔQ–수명 간 음의 상관 관계는 견고하게 유지됨 확인, 모델의 분포 이동 대응력 평가에 최적

In [18]:
batch_cell_frames.append(batch2_cells.copy())
del batch2_cells, batch2_summary, batch2_delta_q
gc.collect()
print('큰 시계열 데이터 메모리 해제 완료')

큰 시계열 데이터 메모리 해제 완료


### 2-3. Batch 3 EDA

#### 기초 EDA

In [19]:
batch3_cells, batch3_summary, batch3_delta_q = load_batch(
    'Batch 3', BATCH_FILES['Batch 3']
)
check_batch('Batch 3', batch3_cells, batch3_summary)

Batch 3 로딩 중...
로드 완료: 셀 46개 / summary 51,007행
Batch 3
셀 수             : 46
Cycle Life 결측   : 2
충전 정책 수      : 8
Cell DataFrame    : (46, 18)
Summary DataFrame : (51007, 11)


,batch,cell_id,cycle_life,charging_policy,first_C_rate,switch_SOC,second_C_rate,newstructure,capacity_95_cycle,delta_q_min,delta_q_var,early_QD_slope,mean_QD,std_QD,mean_IR,mean_Tavg,mean_Tmax,mean_chargetime
0,Batch 3,b3c0,1009.0,5C(67%)-4C-newstructure,5.0,67.0,4.0,1,819.0,-0.023312,0.000057,-0.000015,1.070498,0.000486,0.015229,32.206088,33.580576,10.042507
1,Batch 3,b3c1,1063.0,5.3C(54%)-4C-newstructure,5.3,54.0,4.0,1,851.0,-0.022324,0.000052,-0.000023,1.067775,0.000640,0.014978,34.850694,38.019416,10.041386
2,Batch 3,b3c2,1267.0,5.6C(19%)-4.6C-newstructure,5.6,19.0,4.6,1,966.0,-0.008648,0.000017,-0.000027,1.062083,0.000740,0.014607,36.837682,40.576424,10.043647
3,Batch 3,b3c3,1115.0,5.6C(36%)-4.3C-newstructure,5.6,36.0,4.3,1,886.0,-0.026837,0.000076,-0.000020,1.064698,0.000572,0.015003,36.077997,39.427410,10.041500
4,Batch 3,b3c4,1048.0,5.6C(19%)-4.6C-newstructure,5.6,19.0,4.6,1,799.0,-0.024810,0.000066,-0.000012,1.074605,0.000398,0.015124,32.125576,32.986251,10.045643


,cycle_life,first_C_rate,capacity_95_cycle,delta_q_min,delta_q_var
count,44.000,46.000,46.000,46.000,46.0
mean,1059.659,5.220,839.022,-0.024,0.0
std,313.870,0.535,283.238,0.010,0.0
min,541.000,3.700,254.000,-0.058,0.0
25%,828.000,5.000,669.750,-0.028,0.0
50%,1005.500,5.300,818.500,-0.024,0.0
75%,1155.250,5.600,935.250,-0.019,0.0
max,1935.000,5.900,1609.000,-0.006,0.0


#### Question 1. Cycle Life 분포는 어떻게 생겼는가?
- 150 ~ 2,300 사이클 Histogram
- 장수명(>1,000)/단수명(<500) 비율 확인
- 이상치 셀 식별 - 왜 유독 짧은가?

In [20]:
valid = batch3_cells[batch3_cells.cycle_life.notna()].copy()
short_count = (valid.cycle_life < 500).sum()
long_count = (valid.cycle_life > 1000).sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(valid.cycle_life, bins=np.arange(150, 2351, 100),
             color='steelblue', edgecolor='white')
axes[0].axvline(500, color='tomato', linestyle='--', label='Short < 500')
axes[0].axvline(1000, color='seagreen', linestyle='--', label='Long > 1,000')
axes[0].set(title='Histogram', xlabel='Cycle Life', ylabel='num of Battery Cell')
axes[0].legend()
axes[1].boxplot(valid.cycle_life, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=.6))
axes[1].set(title='Box Plot', ylabel='Cycle Life', xticks=[])
fig.suptitle('Batch 3 - Cycle Life', fontweight='bold')
save_plot(fig, 'batch_3_q1_cycle_life.png')

print(valid.cycle_life.describe().round(1))
print(f'Short: {short_count}개 ({short_count/len(valid):.1%})')
print(f'Long : {long_count}개 ({long_count/len(valid):.1%})')
display(valid.nsmallest(5, 'cycle_life')[
    ['cell_id', 'cycle_life', 'charging_policy', 'delta_q_var']])

# 최단수명 5개 셀 상세 및 전체 median 비교
features = valid.copy()
outlier_cols = [
    'cell_id',
    'cycle_life',
    'charging_policy',
    'first_C_rate',
    'switch_SOC',
    'second_C_rate',
    'delta_q_var',
    'delta_q_min',
    'early_QD_slope',
    'mean_IR',
    'mean_Tavg',
    'mean_chargetime',
]
shortest_cells = features.nsmallest(5, 'cycle_life')[outlier_cols]
print('[Batch 3 최단수명 5개 셀]')
display(shortest_cells)

compare_cols = [
    'delta_q_var',
    'delta_q_min',
    'early_QD_slope',
    'mean_IR',
    'mean_Tavg',
    'mean_chargetime',
]
short_ids = shortest_cells['cell_id']
comparison = pd.DataFrame({
    'shortest_5_median': features[features['cell_id'].isin(short_ids)][compare_cols].median(),
    'batch_median': features[compare_cols].median()
})
print('[Batch 3 최단수명 5개 median vs Batch 전체 median]')
display(comparison.round(5))


저장 완료: results/batch_3_q1_cycle_life.png
count      44.0
mean     1059.7
std       313.9
min       541.0
25%       828.0
50%      1005.5
75%      1155.2
max      1935.0
Name: cycle_life, dtype: float64
Short: 0개 (0.0%)
Long : 23개 (52.3%)


,cell_id,cycle_life,charging_policy,delta_q_var
28,b3c28,541.0,3.7C(31%)-5.9C-newstructure,0.000353
6,b3c6,667.0,3.7C(31%)-5.9C-newstructure,0.000298
31,b3c31,731.0,5.9C(60%)-3.1C-newstructure,0.000145
21,b3c21,772.0,3.7C(31%)-5.9C-newstructure,0.000198
41,b3c41,786.0,5.6C(36%)-4.3C-newstructure,0.000060


[Batch 3 최단수명 5개 셀]


,cell_id,cycle_life,charging_policy,first_C_rate,switch_SOC,second_C_rate,delta_q_var,delta_q_min,early_QD_slope,mean_IR,mean_Tavg,mean_chargetime
28,b3c28,541.0,3.7C(31%)-5.9C-newstructure,3.7,31.0,5.9,0.000353,-0.057831,-2.047446e-04,0.013441,35.722938,10.091233
6,b3c6,667.0,3.7C(31%)-5.9C-newstructure,3.7,31.0,5.9,0.000298,-0.053202,-2.030154e-04,0.013746,34.115818,10.082545
31,b3c31,731.0,5.9C(60%)-3.1C-newstructure,5.9,60.0,3.1,0.000145,-0.035619,-1.799800e-05,0.015873,34.011083,10.045660
21,b3c21,772.0,3.7C(31%)-5.9C-newstructure,3.7,31.0,5.9,0.000198,-0.043276,-1.654836e-04,0.014105,34.927160,10.080266
41,b3c41,786.0,5.6C(36%)-4.3C-newstructure,5.6,36.0,4.3,0.000060,-0.024193,1.185332e-07,0.018296,33.654851,10.043180


[Batch 3 최단수명 5개 median vs Batch 전체 median]


,shortest_5_median,batch_median
delta_q_var,0.00020,0.00007
delta_q_min,-0.04328,-0.02479
early_QD_slope,-0.00017,-0.00002
mean_IR,0.01411,0.01537
mean_Tavg,34.11582,34.12338
mean_chargetime,10.08027,10.04325


#### Question 2. 열화 곡선 - 방전 용량이 어떻게 감소하는가?
- 사이클 별 Qd 추이 시각화
- 열화 속도가 일정한가, 가속되는가?
- Knee point - 급격한 열화 시작점 탐색

In [21]:
valid_summary = batch3_summary[batch3_summary.cycle_life.notna()].copy()
cell_ids = valid_summary.cell_id.unique()
colors = cm.coolwarm(np.linspace(0, 1, len(cell_ids)))
nominal = valid_summary[valid_summary.cycle.between(10, 20)].groupby('cell_id').QD.median().median()

fig, ax = plt.subplots(figsize=(14, 6))
for color, cell_id in zip(colors, cell_ids):
    one_cell = valid_summary[
        (valid_summary.cell_id == cell_id)
        & valid_summary.QD.between(nominal * .7, nominal * 1.2)
    ]
    ax.plot(one_cell.cycle, one_cell.QD, color=color, linewidth=.8, alpha=.7)
ax.axhline(nominal * .8, color='red', linestyle='--', label='EOL (80%)')
ax.set(title='Batch 3 - QD Curve', xlabel='Cycle', ylabel='QD (Ah)', ylim=(.7, 1.2))
ax.legend()
save_plot(fig, 'batch_3_q2_degradation.png')

print('용량 95% 도달 시점 (capacity_95_cycle) 요약')
display(batch3_cells.loc[batch3_cells.cycle_life.notna(), 'capacity_95_cycle'].describe().round(1))

# 실제 Piecewise Knee 검출 분석
knee_rows = []
for cell_id, one_cell in batch3_summary.groupby('cell_id'):
    result = detect_piecewise_knee(one_cell)
    result['cell_id'] = cell_id
    knee_rows.append(result)
knee_df = pd.DataFrame(knee_rows)
knee_df = knee_df.merge(
    batch3_cells[['cell_id', 'cycle_life']],
    on='cell_id',
    how='left')
valid_knee = knee_df.dropna(
    subset=['knee_cycle', 'cycle_life']).copy()
valid_knee['knee_ratio'] = (
    valid_knee['knee_cycle']
    / valid_knee['cycle_life'])

if len(valid_knee) > 1:
    rho = valid_knee[['knee_cycle', 'cycle_life']].corr(method='spearman').iloc[0, 1]
    print(f'[Batch 3] Knee vs Cycle Life rho: {rho:.3f}')
    print(f'[Batch 3] Median Knee Cycle: {valid_knee.knee_cycle.median():.1f}')
    print(f'[Batch 3] Median Knee Ratio: {valid_knee.knee_ratio.median():.3f}')
    print(f'[Batch 3] Median Pre-Knee Slope: {valid_knee.pre_knee_slope.median():.6f}')
    print(f'[Batch 3] Median Post-Knee Slope: {valid_knee.post_knee_slope.median():.6f}')
    print(f'[Batch 3] Median Slope Ratio: {valid_knee.slope_ratio.median():.2f}x')


저장 완료: results/batch_3_q2_degradation.png
용량 95% 도달 시점 (capacity_95_cycle) 요약


count      44.0
mean      805.2
std       238.6
min       254.0
25%       665.5
50%       808.5
75%       889.5
max      1374.0
Name: capacity_95_cycle, dtype: float64

[Batch 3] Knee vs Cycle Life rho: 0.980
[Batch 3] Median Knee Cycle: 819.0
[Batch 3] Median Knee Ratio: 0.793
[Batch 3] Median Pre-Knee Slope: -0.000061
[Batch 3] Median Post-Knee Slope: -0.000684
[Batch 3] Median Slope Ratio: 9.97x


#### Question 3. ΔQ(V) 곡선 - 초기 사이클에서 차이가 보이는가?
- 사이클 100번 - 사이클 10번의 Q(V) 차이 계산
- 장수명 셀 vs 단수명 셀의 ΔQ 형태 비교
- 이를 구분할 수 있는 통계값으로 피쳐 추출

In [22]:
life_group = batch3_cells[batch3_cells.cycle_life.notna()].copy()
life_group['life_group'] = 'middle'
life_group.loc[life_group.cycle_life < 500, 'life_group'] = 'short'
life_group.loc[life_group.cycle_life > 1000, 'life_group'] = 'long'

if (life_group.life_group == 'short').sum() < 2 or (life_group.life_group == 'long').sum() < 2:
    low, high = life_group.cycle_life.quantile([.25, .75])
    # 기존 absolute label 초기화
    life_group['life_group'] = 'middle'
    life_group.loc[life_group.cycle_life <= low, 'life_group'] = 'short'
    life_group.loc[life_group.cycle_life >= high, 'life_group'] = 'long'
    print(f'절대 기준 표본 부족 → 하위 25%(≤{low:.0f})와 상위 25%(≥{high:.0f}) 비교')

voltage = np.linspace(2, 3.6, 1000)
fig, ax = plt.subplots(figsize=(10, 5))
for group, color in [('short', 'tomato'), ('long', 'seagreen')]:
    ids = life_group.loc[life_group.life_group == group, 'cell_id']
    curves = np.vstack([batch3_delta_q[cell_id] for cell_id in ids])
    ax.plot(voltage, curves.mean(axis=0), color=color, label=f'{group} (n={len(ids)})')
ax.set(title='Batch 3 - Delta Q(V)', xlabel='Voltage (V)',
       ylabel='Qd(100) - Qd(10) (Ah)')
ax.legend()
save_plot(fig, 'batch_3_q3_delta_q.png')
display(life_group.groupby('life_group')[['delta_q_min', 'delta_q_var']].median().round(6))


절대 기준 표본 부족 → 하위 25%(≤828)와 상위 25%(≥1155) 비교
저장 완료: results/batch_3_q3_delta_q.png


,delta_q_min,delta_q_var
life_group,,
long,-0.017059,0.000035
middle,-0.025985,0.000071
short,-0.027859,0.000083


#### Question 4. 충전 조건 (C-rate)과 수명의 관계는?
- 충전 프로토콜별 평균 수명 비교
- 고속 충전 셀이 정말 수명이 짧은가?
- 충전 전류 패턴과 열화 속도 상관 분석

In [23]:
valid = batch3_cells[batch3_cells.cycle_life.notna()].copy()
features = valid.copy()

policy_summary = (
    features
    .groupby('charging_policy')['cycle_life']
    .agg([
        'count',
        'mean',
        'median',
        'std',
        'min',
        'max'
    ])
    .sort_values('median', ascending=False)
)
policy_summary.to_csv(RESULT_DIR / 'batch_3_policy_summary.csv')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar(range(len(policy_summary)), policy_summary['mean'], color='steelblue', alpha=.75)
axes[0].set_xticks(range(len(policy_summary)))
axes[0].set_xticklabels(policy_summary.index, rotation=45, ha='right', fontsize=8)
axes[0].set(title='Average Cycle Life by Policy', ylabel='mean(Cycle Life)')
axes[1].scatter(valid.first_C_rate, valid.cycle_life,
                color='steelblue', alpha=.7, edgecolors='white')
corr = valid.first_C_rate.corr(valid.cycle_life, method='spearman')
axes[1].set(title=f'First C-rate vs Cycle Life (rho={corr:.2f})',
            xlabel='First C-rate', ylabel='Cycle Life')
fig.suptitle('Batch 3 - Charging Policy', fontweight='bold')
save_plot(fig, 'batch_3_q4_charge_policy.png')

print('[Batch 3 Charging Policy Summary]')
display(policy_summary.round(1))

# C-rate / SOC 분석
charge_features = [
    'first_C_rate',
    'switch_SOC',
    'second_C_rate',
]
target_features = [
    'cycle_life',
    'delta_q_var',
    'early_QD_slope',
]
charge_corr = (
    features[
        charge_features + target_features
    ]
    .corr(method='spearman')
    .loc[
        charge_features,
        target_features
    ]
)
print('[Batch 3 Charging Condition Correlation]')
display(charge_corr.round(3))

# newstructure 분석
structure_summary = (
    features
    .groupby('newstructure')
    .agg(
        n=('cell_id', 'count'),
        median_cycle_life=('cycle_life', 'median'),
        mean_cycle_life=('cycle_life', 'mean'),
        median_delta_q_var=('delta_q_var', 'median'),
        median_early_qd_slope=('early_QD_slope', 'median'),
    )
)
print('[Batch 3 newstructure Summary]')
display(structure_summary.round(4))


저장 완료: results/batch_3_q4_charge_policy.png
[Batch 3 Charging Policy Summary]


,count,mean,median,std,min,max
charging_policy,,,,,,
4.8C(80%)-4.8C-newstructure,6,1564.2,1640.0,285.7,1078.0,1836.0
5.3C(54%)-4C-newstructure,8,1074.4,1051.0,129.7,935.0,1315.0
5.6C(19%)-4.6C-newstructure,8,1001.4,1038.0,174.6,796.0,1267.0
5C(67%)-4C-newstructure,7,1105.7,1009.0,402.9,813.0,1935.0
5.6C(36%)-4.3C-newstructure,8,930.9,890.5,135.0,786.0,1155.0
5.9C(15%)-4.6C-newstructure,2,867.0,867.0,12.7,858.0,876.0
5.9C(60%)-3.1C-newstructure,2,866.5,866.5,191.6,731.0,1002.0
3.7C(31%)-5.9C-newstructure,3,660.0,667.0,115.7,541.0,772.0


[Batch 3 Charging Condition Correlation]


,cycle_life,delta_q_var,early_QD_slope
first_C_rate,-0.229,0.289,-0.123
switch_SOC,0.443,-0.450,0.437
second_C_rate,0.016,-0.077,-0.230


[Batch 3 newstructure Summary]


,n,median_cycle_life,mean_cycle_life,median_delta_q_var,median_early_qd_slope
newstructure,,,,,
1,44,1005.5,1059.6591,0.0001,-0.0


#### Question 5. 상관관계 - 어떤 신호가 수명과 연관되어 있는가?
- 초기 사이클 피켜들과 Cycle Life 상관 계수 확인
- 가장 강한 관계 식별
- 멀티클리니어리티 문제 확인

In [24]:
valid_summary = batch3_summary[batch3_summary.cycle_life.notna()].copy()
early_features = valid_summary[valid_summary.cycle.between(10, 100)].groupby('cell_id').agg(
    cycle_life=('cycle_life', 'first'),
    mean_QD=('QD', 'mean'),
    std_QD=('QD', 'std'),
    mean_IR=('IR', 'mean'),
    mean_Tavg=('Tavg', 'mean'),
    mean_Tmax=('Tmax', 'mean'),
    mean_chargetime=('chargetime', 'mean'),
)
extra_features = batch3_cells.set_index('cell_id')[['delta_q_min', 'delta_q_var', 'first_C_rate']]
early_features = early_features.join(extra_features)
corr_matrix = early_features.corr(method='spearman')
early_features.to_csv(RESULT_DIR / 'batch_3_early_features.csv')
corr_matrix.to_csv(RESULT_DIR / 'batch_3_correlation.csv')

fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(image, ax=ax)
labels = corr_matrix.columns.tolist()
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticklabels(labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        value = corr_matrix.iloc[i, j]
        ax.text(j, i, f'{value:.2f}', ha='center', va='center', fontsize=8,
                color='white' if abs(value) > .5 else 'black')
ax.set_title('Batch 3 - Early 100 Cycle Correlation', fontweight='bold')
save_plot(fig, 'batch_3_q5_correlation.png')

print('[Cycle Life 상관 순위]')
display(corr_matrix.cycle_life.drop('cycle_life').sort_values(key=abs, ascending=False).round(3))

print('[피처끼리 |rho| > 0.9인 조합: 다중공선성 후보]')
pairs = []
feature_corr = corr_matrix.drop(index='cycle_life', columns='cycle_life')
for i in range(len(feature_corr.columns)):
    for j in range(i + 1, len(feature_corr.columns)):
        value = feature_corr.iloc[i, j]
        if abs(value) > .9:
            pairs.append((feature_corr.columns[i], feature_corr.columns[j], round(value, 3)))
display(pd.DataFrame(pairs, columns=['feature_1', 'feature_2', 'rho']))

# 3-Batch Correlation 비교표 생성
all_features = pd.concat(
    batch_cell_frames + [batch3_cells],
    ignore_index=True
)

feature_cols = [
    'delta_q_var',
    'delta_q_min',
    'early_QD_slope',
    'mean_QD',
    'std_QD',
    'mean_IR',
    'mean_Tavg',
    'mean_Tmax',
    'mean_chargetime',
    'first_C_rate',
]
corr_rows = []
for batch, group in all_features.groupby('batch'):
    for feature in feature_cols:
        rho = group[feature].corr(
            group['cycle_life'],
            method='spearman'
        )
        corr_rows.append({
            'batch': batch,
            'feature': feature,
            'rho': rho,
        })
corr_compare = (
    pd.DataFrame(corr_rows)
    .pivot(
        index='feature',
        columns='batch',
        values='rho'
    )
)
print('[Batch 1 / Batch 2 / Batch 3 Feature Correlation with Cycle Life]')
display(corr_compare.round(3))


저장 완료: results/batch_3_q5_correlation.png
[Cycle Life 상관 순위]


delta_q_var       -0.797
delta_q_min        0.776
first_C_rate      -0.229
mean_chargetime    0.202
mean_IR            0.167
mean_QD            0.163
mean_Tmax         -0.037
std_QD            -0.035
mean_Tavg          0.025
Name: cycle_life, dtype: float64

[피처끼리 |rho| > 0.9인 조합: 다중공선성 후보]


,feature_1,feature_2,rho
0,mean_Tavg,mean_Tmax,0.976
1,delta_q_min,delta_q_var,-0.984


[Batch 1 / Batch 2 / Batch 3 Feature Correlation with Cycle Life]


batch,Batch 1,Batch 2,Batch 3
feature,,,
delta_q_min,0.850,0.661,0.776
delta_q_var,-0.871,-0.709,-0.797
early_QD_slope,0.478,-0.271,0.184
first_C_rate,-0.483,0.055,-0.229
mean_IR,0.136,-0.343,0.167
mean_QD,0.203,0.095,0.163
mean_Tavg,-0.371,0.172,0.025
mean_Tmax,-0.322,-0.126,-0.037
mean_chargetime,0.617,-0.391,0.202


#### Question별 핵심 해석 및 시사점

| Question | 확인한 핵심 내용 | 해석 | 모델링 시사점 |
|---|---|---|---|
| **Q1. Cycle Life 분포** | • 44개 셀 유효 (2개 결측)<br>• 중앙값 **1,005.5 cycle**, 범위 **541~1,935** (장수명 `>1,000` 52.3%, 단수명 `<500` 0%)<br>• 최단수명 5개 셀의 `delta_q_var` 중앙값은 배치 중앙값의 약 **3.3배** | • 3개 배치 중 가장 장수명 중심 및 높은 분산<br>• 최단명 셀 공통 정책은 표본 부족(3개)으로 인과관계 단정 불가 | • 장수명 영역 및 넓은 수명 범위에 대한 **추가 일반화 테스트셋**으로 활용<br>• 극단적 장수명 셀 오차는 RMSE 및 개별 잔차 분석으로 모니터링 |
| **Q2. 열화 곡선과 Knee** | • 초기 완만 구간 장기화 (용량 95% 도달 중앙값 **808.5 cycle**, Knee 중앙값 **819 cycle**)<br>• Knee 발생 시점은 전체 수명의 약 **79.3%** 지점 (수명 상관 0.980)<br>• Knee 이후 열화 기울기 절댓값 약 **9.97배** 가속 | • 가속 열화 시작 시점이 늦으나, 후반부 급격한 열화 패턴은 전 배치 공통 | • 초기 100 사이클 내 미세 변화 포착 피처(ΔQ 통계량) 필수<br>• Knee 피처는 미래 정보 누수로 설명 및 사후 검증용으로만 한정 |
| **Q3. ΔQ(V)** | • 하위 25%(상대 단수명) vs 상위 25%(상대 장수명) 비교<br>• `delta_q_var` 중앙값 약 **2.4배** 차이 (0.000083 vs 0.000035)<br>• 2.6 V 부근에서 곡선 간격 최대 | • 배치 1·2 대비 그룹 간 격차는 작으나 **"초기 ΔQ 변화 적을수록 수명 증가"** 방향성 동일 | • `delta_q_var`를 핵심 피처로 유지<br>• 신호 격차가 완만한 특성을 고려해 충전조건·IR·온도 등 보조 피처 결합 검토 |
| **Q4. 충전 조건** | • 정책별 평균 수명 **660.0~1,564.2 cycle**<br>• 수명 상관: 첫 C-rate **-0.229**, 전환 SOC **+0.443**, 두 번째 C-rate **+0.016**<br>• 최장수명 정책은 고속 충전군(`4.8C(80%)-4.8C-newstructure`) | • 단순 C-rate 단독 효과가 아닌 전환 SOC·두 번째 C-rate·셀 구조의 복합 영향 | • 정책 범주명 대신 세부 수치(C-rate, SOC) 및 `newstructure` 플래그로 분해 적용<br>• 비선형 상호작용은 트리 계열 모델로 보조 검증 |
| **Q5. 상관관계** | • 수명 상관: `delta_q_var` **-0.797**, `delta_q_min` **+0.776**으로 최상위<br>• 용량 편차(`std_QD` -0.035) 및 온도 변수(상관 ~0) 설명력 미미<br>• 두 온도 피처 간 강한 다중공선성 (상관 0.976) | • ΔQ 신호의 강력한 일관성 재확인, 온도 단독 설명력은 부재 | • `delta_q_var` 1순위 피처 고정<br>• 다중공선성 방지를 위해 온도는 1개만 선별 사용, 기타 피처는 ablation 검증 |

**Batch 3 결론:**  
장수명 및 넓은 수명 분포에서도 ΔQ–수명 간 강한 음의 상관 일관성 확인, 충전 조건은 단독 C-rate보다 세부 조건 분해 및 조합 접근 필요

In [25]:
batch_cell_frames.append(batch3_cells.copy())
del batch3_cells, batch3_summary, batch3_delta_q
gc.collect()
print('큰 시계열 데이터 메모리 해제 완료')

큰 시계열 데이터 메모리 해제 완료


# 3. Batch별 특성 및 비교

대용량 시계열 메모리 해제 후 셀 단위 요약 데이터 취합 및 3개 배치 분포 비교 분석

In [26]:
all_cells = pd.concat(batch_cell_frames, ignore_index=True)

batch_compare = all_cells.groupby('batch').agg(
    total_cells=('cell_id', 'size'),
    valid_target=('cycle_life', 'count'),
    median_life=('cycle_life', 'median'),
    min_life=('cycle_life', 'min'),
    max_life=('cycle_life', 'max'),
    mean_delta_q_var=('delta_q_var', 'mean'),
)
batch_compare['short_ratio'] = all_cells.groupby('batch').cycle_life.apply(
    lambda values: (values.dropna() < 500).mean()
)
batch_compare['long_ratio'] = all_cells.groupby('batch').cycle_life.apply(
    lambda values: (values.dropna() > 1000).mean()
)

all_cells.to_csv(RESULT_DIR / 'all_batch_cell_features.csv', index=False)
batch_compare.to_csv(RESULT_DIR / 'batch_comparison.csv')
display(batch_compare.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
groups = [
    all_cells.loc[(all_cells.batch == name) & all_cells.cycle_life.notna(), 'cycle_life']
    for name in BATCH_FILES
]
axes[0].boxplot(groups, tick_labels=list(BATCH_FILES), patch_artist=True)
axes[0].set(title='Cycle Life by Batch', ylabel='Cycle Life')

for name, color in zip(BATCH_FILES, ['steelblue', 'tomato', 'seagreen']):
    values = all_cells.loc[(all_cells.batch == name) & all_cells.cycle_life.notna(), 'cycle_life']
    axes[1].hist(values, bins=np.arange(150, 2351, 100), alpha=.4, label=name, color=color)
axes[1].set(title='Cycle Life Distribution', xlabel='Cycle Life', ylabel='Cells')
axes[1].legend()
save_plot(fig, 'batch_comparison_cycle_life.png')

,total_cells,valid_target,median_life,min_life,max_life,mean_delta_q_var,short_ratio,long_ratio
batch,,,,,,,,
Batch 1,46,46,858.5,534.0,1227.0,0.0,0.000,0.217
Batch 2,47,39,472.0,392.0,1186.0,0.0,0.718,0.077
Batch 3,46,44,1005.5,541.0,1935.0,0.0,0.000,0.523


저장 완료: results/batch_comparison_cycle_life.png


| 비교 항목 | Batch 1 | Batch 2 | Batch 3 |
|---|---:|---:|---:|
| 유효 Target | 46 | 39 | 44 |
| Cycle Life 중앙값 | 858.5 | 472.0 | 1,005.5 |
| 단수명(`<500`) | 0.0% | 71.8% | 0.0% |
| 장수명(`>1,000`) | 21.7% | 7.7% | 52.3% |
| 첫 C-rate 상관 | -0.483 | +0.055 | -0.229 |
| ΔQ variance 상관 | -0.871 | -0.709 | -0.797 |

배치별 수명 분포 차이 존재하나, 초기 ΔQ variance와 Cycle Life 간 강력한 음의 상관관계는 3개 배치 전반에서 일관되게 확인

# 4. EDA 기반 인사이트

1. **Batch Shift 존재**: Batch 2는 단수명 중심, Batch 3는 장수명 중심이므로 전체 무작위 분할(Random split) 시 일반화 성능 과대평가 위험 존재
2. **ΔQ(V)의 일관된 조기 신호**: `delta_q_var`와 Cycle Life 간 상관계수가 3개 배치 모두 -0.7 이하로 가장 강력하고 일관된 핵심 피처
3. **충전 조건의 복합 작용**: 첫 C-rate 단독 인과관계 미성립, 2차 C-rate, 전환 SOC, 셀 구조(`newstructure`)의 복합 작용 고려 필요
4. **주요 피처 간 다중공선성**: `delta_q_min`–`delta_q_var`, `mean_Tavg`–`mean_Tmax` 등 중복 신호는 대표 피처 선별 또는 정규화 규제 모델 적용 필요
5. **Knee Point의 미래정보 누수**: 전체 열화 곡선 관측 후 산출되는 사후 지표이므로 조기 예측 피처에서 제외, 메커니즘 설명 및 검증용으로만 한정

In [27]:
assert len(all_cells) == 139
assert all_cells.cycle_life.notna().sum() == 129
assert all_cells.batch.nunique() == 3
assert len(list(RESULT_DIR.glob('batch_*.png'))) == 16
print('검증 완료: 3개 Batch, 139개 셀, 유효 Target 129개, 그래프 16개')


검증 완료: 3개 Batch, 139개 셀, 유효 Target 129개, 그래프 16개
